# 03 — Preparação dos dados e engenharia de atributos

Este notebook documenta **o que entra no modelo** em cada cenário, como cada variável é transformada e como os dados são divididos no tempo. Código: `src/config.py`, `src/data_processing.py` e `src/feature_engineering.py`.

In [1]:
# Configuração: roda a partir da raiz do repositório
import os, sys, warnings
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
warnings.filterwarnings("ignore")
import json
import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
from src import config

In [2]:
from src import data_processing as dp, feature_engineering as fe, train
base = dp.carregar_base()

## 1. Cenários de informação (prevenção de vazamento)

| Cenário | Momento simulado | O que entra |
|---|---|---|
| **A (principal)** | chamada chega à PRF | tempo, local, via, clima, tipo de acidente, nº de veículos |
| B | antes de saber o que aconteceu | só tempo, local, via e clima |
| C | depois da perícia (**não é previsão**) | A + causa presumível |
| A + pessoas | teste de sensibilidade | A + nº de pessoas envolvidas |

`mortos`, `feridos`, `feridos_leves`, `feridos_graves`, `ilesos` e `ignorados` **definem** o alvo e nunca entram. O teste `tests/test_pipeline.py` falha se alguma delas for usada.

In [3]:
for c in config.CENARIOS:
    print(f"{c:10s} ({len(config.features_do_cenario(c))} colunas):", config.features_do_cenario(c))

B          (30 colunas): ['uf', 'regiao', 'br', 'dia_semana', 'fase_dia', 'condicao_meteo', 'tipo_pista', 'uso_solo', 'sentido_via', 'faixa_horaria', 'hora_sin', 'hora_cos', 'mes', 'km', 'latitude', 'longitude', 'fim_de_semana', 'feriado', 'vespera_ou_pos_feriado', 'trc_reta', 'trc_curva', 'trc_intersecao', 'trc_ponte', 'trc_viaduto', 'trc_rotatoria', 'trc_retorno', 'trc_tunel', 'trc_desvio', 'trc_nao_informado', 'trecho']
A          (32 colunas): ['uf', 'regiao', 'br', 'dia_semana', 'fase_dia', 'condicao_meteo', 'tipo_pista', 'uso_solo', 'sentido_via', 'faixa_horaria', 'tipo_acidente', 'hora_sin', 'hora_cos', 'mes', 'km', 'latitude', 'longitude', 'fim_de_semana', 'feriado', 'vespera_ou_pos_feriado', 'trc_reta', 'trc_curva', 'trc_intersecao', 'trc_ponte', 'trc_viaduto', 'trc_rotatoria', 'trc_retorno', 'trc_tunel', 'trc_desvio', 'trc_nao_informado', 'veiculos', 'trecho']
C          (33 colunas): ['uf', 'regiao', 'br', 'dia_semana', 'fase_dia', 'condicao_meteo', 'tipo_pista', 'uso_solo',

In [4]:
# Checagem: alvo é função determinística das contagens (por isso as contagens são proibidas)
chk = pd.crosstab(base[config.TARGET].map(dict(enumerate(config.CLASSES_CURTAS))),
                  [np.where(base.mortos > 0, "mortos>0", "mortos=0"), np.where(base.feridos > 0, "feridos>0", "feridos=0")])
chk

col_0        mortos=0            mortos>0          
col_1       feridos=0 feridos>0 feridos=0 feridos>0
gravidade                                          
Fatais              0         0     27148     16271
Feridas             0    462732         0         0
Sem vítimas    126269         0         0         0

## 2. Engenharia de atributos

Criadas linha a linha em `data_processing.limpar` (não aprendem nada dos dados):

- **tempo:** hora em seno/cosseno (23h fica perto de 0h), faixa horária, mês, fim de semana, feriado nacional e véspera/dia seguinte (pacote `holidays`);
- **via:** indicadores binários extraídos do `tracado_via` multivalorado;
- **local:** região a partir da UF e **trecho** = UF + BR + faixa de 10 km.

Aprendidas dentro do `Pipeline` (ajustadas só no treino):

- **target encoding do trecho** com *cross fitting* (5 folds), que gera a taxa suavizada de cada classe no trecho sem usar a própria linha;
- **One-Hot + padronização** (Regressão Logística) ou **codificação ordinal** (árvores), agrupando categorias com menos de 50 casos.

In [5]:
base[["hora", "hora_sin", "hora_cos", "faixa_horaria", "feriado", "vespera_ou_pos_feriado", "trecho"] + config.TRACADO_COLS[:4]].head()

,hora,hora_sin,hora_cos,faixa_horaria,feriado,vespera_ou_pos_feriado,trecho,trc_reta,trc_curva,trc_intersecao,trc_ponte
0,0.0,0.0,1.0,madrugada,1,0,RJ-40-110,1,0,0,0
1,0.0,0.0,1.0,madrugada,1,0,PR-153-50,1,0,0,0
2,0.0,0.0,1.0,madrugada,1,0,GO-153-430,1,0,0,0
3,0.0,0.0,1.0,madrugada,1,0,SC-280-70,0,0,0,0
4,0.0,0.0,1.0,madrugada,1,0,AC-364-130,1,0,0,0


## 3. Divisão temporal

In [6]:
tr, va, te = train.dividir(base)
div = pd.DataFrame({
    "anos": [f"{config.ANOS_TREINO[0]}-{config.ANOS_TREINO[-1]}", str(config.ANO_VALIDACAO), str(config.ANO_TESTE)],
    "acidentes": [len(tr), len(va), len(te)],
}, index=["treino", "validação", "teste"])
for k, nome in enumerate(config.CLASSES_CURTAS):
    div[f"% {nome}"] = [round(100 * (d[config.TARGET] == k).mean(), 1) for d in (tr, va, te)]
div

,anos,acidentes,% Sem vítimas,% Feridas,% Fatais
treino,2019-2023,327879,17.5,75.4,7.1
validação,2024,73155,16.1,76.8,7.1
teste,2025,72528,15.4,77.5,7.2


Por que não um split aleatório? Acidentes do mesmo trecho e do mesmo mês cairiam no treino e no teste, e o modelo seria avaliado num cenário mais fácil que o real. O notebook 06 mede essa diferença.

## 4. O pré-processamento em ação (amostra do treino)

In [7]:
amostra = tr.sample(20000, random_state=config.SEED)
pre_lin = fe.preprocessador_linear("A").fit(amostra[config.features_do_cenario("A")], amostra[config.TARGET])
pre_arv = fe.preprocessador_arvores("A").fit(amostra[config.features_do_cenario("A")], amostra[config.TARGET])
print("Regressão Logística ->", len(pre_lin.get_feature_names_out()), "colunas após One-Hot")
print("Árvores             ->", len(pre_arv.get_feature_names_out()), "colunas (ordinal + TE)")
fe.nomes_legiveis(pre_arv.get_feature_names_out())[-6:]

Regressão Logística -> 153 colunas após One-Hot
Árvores             -> 34 colunas (ordinal + TE)


['trc_desvio',
 'trc_nao_informado',
 'veiculos',
 'trecho→P(sem vítimas)',
 'trecho→P(feridas)',
 'trecho→P(fatais)']

## 5. Desbalanceamento

A classe Fatais é ~7% dos acidentes. Estratégias (escolhidas na validação, notebook 05):

1. **pesos de amostra:** `balanceado` (inverso da frequência), `raiz` (meio-termo) ou `nenhum`;
2. **ajuste do limiar de decisão:** multiplicadores por classe aplicados às probabilidades, escolhidos em 2024 para maximizar o F1-macro (`fe.AjusteLimiar`).

SMOTE não foi usado: as variáveis são quase todas categóricas, e exemplos sintéticos distorcem as probabilidades.